# Introduction to ensemble methods

In this practical, you will construct your own bagging ensemble, to boost performance on a binary classification problem.


## The problem

Users of your service sometimes close their accounts, even though they have been with you for a while. It would be useful if you could predict which ones are likely to do this, before it happens, so you can take action to retain their custom.

You have access to 20 different features for each user, based on data such as how often they log in, how many purchases they make, etc. The aim is to see if you can predict whether a user will end up closing their account or not.

## Getting started

First, load the data from `account_data.csv` using `pandas` into a DataFrame called `data`. 

Find out how many users left the service by examining the `closed_account` column. 

*Hint: use the `.value_counts()` method*

In [17]:
import pandas as pd

# Your code here...
data=pd.read_csv('data/account_data.csv')

data.head()

,metric_1,metric_2,metric_3,metric_4,metric_5,metric_6,metric_7,metric_8,metric_9,metric_10,...,metric_12,metric_13,metric_14,metric_15,metric_16,metric_17,metric_18,metric_19,metric_20,closed_account
0,-0.940182,0.207301,0.528861,1.049908,-0.835118,-0.400130,-0.394802,2.187548,-0.572739,0.788069,...,0.481080,-0.141954,0.222494,0.949518,0.831708,-2.298094,1.090053,0.726359,2.937407,0
1,-0.199183,0.655927,1.764300,0.517764,1.340340,0.006723,-0.606941,1.174000,-1.829904,-1.511114,...,0.710257,0.076678,0.197514,-1.377654,0.836603,-1.270138,-1.323004,0.590826,0.444392,0
2,0.263755,1.565332,-1.226079,0.253425,0.718595,-0.018978,0.127827,0.650657,1.930543,-1.240717,...,-0.810745,0.652507,-0.566913,1.819934,0.080513,1.017901,0.131095,-0.119743,-0.078145,1
3,-1.782073,-2.691368,-0.789498,1.472354,0.026098,-0.800538,-1.412452,-0.195018,0.674284,-0.823797,...,-1.432476,-0.322312,0.915371,-0.543112,-1.185113,-0.343656,-0.121278,-0.209454,1.395878,0
4,0.076881,-1.981506,-0.206010,-0.085352,-1.014990,0.446084,-1.443363,0.492055,-0.941904,1.613225,...,-0.728380,0.027557,0.852682,-0.884311,2.079615,-0.056549,0.653112,1.443499,0.074054,1


In [18]:
data['closed_account'].value_counts()

closed_account
1    102
0     98
Name: count, dtype: int64

## Setting up for classification

Before proceeding, we will split the data into training and testing subsets.

Create two variables, `X` and `y`, from the `data` DataFrame.

`X` should be all the columns that you want to use in your prediction (ie. all except for `closed_account`)

`y` should be the column that you are trying to predict (ie. `closed_account`)

These are the conventional variable names used in machine learning to represent your predicting features (`X`) and the values you are trying to predict (`y`).

In [19]:
# Your code here...

X=data.drop('closed_account',axis=1)
y=data['closed_account']


Models will have access to the training set in order to learn, but we will evaluate them on the unseen test set.

Use `train_test_split` from `sklearn.model_selection` to split the data up. 

This function takes your `X` and `y` (our 20 monthly data points, and the closed/not closed label) and splits them up. It returns four lists - one `X` for each of train/test and the same for `y`.

The default gives a ratio of 3:1 train/test split.

Call the new variables `X_train`, `X_test`, `y_train`, and `y_test`.

(Set `random_state` to `5`, so that results are the same every time!)

In [20]:
from sklearn.model_selection import train_test_split

# Your code here...
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=5)

## Single model performance

Before trying an ensemble, we need some baselines in order to determine whether the ensemble method actually improved anything.

We will use a simple Naive Bayes classifier here with default parameters.

Import the required class and instantiate it with the variable name `model`.


In [21]:
from sklearn.naive_bayes import GaussianNB

# Your code here...
model=GaussianNB()

Use the `.fit()` method of the model to train it on the training subset of the data.

Use the `.score()` method the same way, using the same data, to see how well it performs on the data it learned from.

Use `model.predict()` to generate predictions for `X_train` and compare these predictions to the ground truth (`y_train`) using a classification report.

In [22]:
from sklearn.metrics import classification_report

# Your code here...
model.fit(X_train,y_train)
y_pred=model.predict(X_train)
print(model.score(X_train,y_train))
print(classification_report(y_train,y_pred))


0.6733333333333333
              precision    recall  f1-score   support

           0       0.65      0.60      0.63        68
           1       0.69      0.73      0.71        82

    accuracy                           0.67       150
   macro avg       0.67      0.67      0.67       150
weighted avg       0.67      0.67      0.67       150



Repeat the above scoring process but now use the unseen test data.

Do you expect accuracy to change? If so, in what direction?

What are your thoughts on how well the model performs on the two classes?

In [23]:
# Your code here...
y_pred=model.predict(X_test)
print(model.score(X_test,y_test))
print(classification_report(y_test,y_pred))

0.44
              precision    recall  f1-score   support

           0       0.54      0.43      0.48        30
           1       0.35      0.45      0.39        20

    accuracy                           0.44        50
   macro avg       0.44      0.44      0.44        50
weighted avg       0.46      0.44      0.45        50



## Bagging ensemble

Now let's see what effect a simple bagging ensemble has.

`sklearn` has a useful utility class that will handle this for us: `sklearn.ensemble.BaggingClassifier`

This has several parameters which can be tuned but the main ones are:

`estimator` - the model to use

`n_estimators` - how many models to use. Default is 10.

`random_state` - ensures the same results every time

Create a bagging classifier named `ensemble`, using `GaussianNB()` as the `estimator`. Keep `n_estimators` at 10 and set `random_state` to 5.

In [24]:
from sklearn.ensemble import BaggingClassifier

# Your code here...
model=GaussianNB()
ensemble=BaggingClassifier(estimator=model,n_estimators=10,random_state=5)


Once you have created this meta-model, it works exactly the same way as any model in `sklearn` - it has  methods like `.fit()` and `.score()` and `.predict()`.

Repeat the training/scoring/classification report process from the single model approach you did previously.

Train only on the training data and test only on the test data.

What do you observe now?

In [25]:
# Your code here...
ensemble.fit(X_train,y_train)
y_pred=ensemble.predict(X_test)
print(ensemble.score(X_test,y_test))
print(classification_report(y_test,y_pred))

0.58
              precision    recall  f1-score   support

           0       0.70      0.53      0.60        30
           1       0.48      0.65      0.55        20

    accuracy                           0.58        50
   macro avg       0.59      0.59      0.58        50
weighted avg       0.61      0.58      0.58        50



## Conclusions

Using an ensemble of methods improved performance in a binary classification task by a good margin, though performance was still generally quite low.

This may be because the signal to noise ratio in the data is low (we have used synthetic data), meaning the model needs to make predictions with very little information. Even so, what could you investigate in order to improve performance on this task?

In [26]:
# Your code below
